## Parte 1 - Extracción y Diagnóstico (Extract)

In [0]:
## Cargue ambos archivos CSV en DataFrames de Pandas (df_restaurantes y df_menus)

import pandas as pd
df_restaurantes = pd.read_csv("/Workspace/Users/josevalebboy@gmail.com/JOSE_VALERIANO-PROYECTO-ETL-DELIVERY/PROYECTO_FINAL (1)/restaurants.csv/restaurants.csv")

In [0]:
import pandas as pd
import glob

archivos = glob.glob( "/Workspace/Users/josevalebboy@gmail.com/JOSE_VALERIANO-PROYECTO-ETL-DELIVERY/PROYECTO_FINAL (1)/restaurant-menus.csv/*.csv")

df_menus = pd.concat([pd.read_csv(f) for f in archivos],
            ignore_index=True
)

In [0]:
## Realice un análisis exploratorio (Data Profiling): Imprima las dimensiones (.shape), los tipos de datos (.dtypes) y la cantidad de valores nulos por columna de ambos DataFrames.

print("Restaurantes:")
print(df_restaurantes.shape)

print("Menus:")
print(df_menus.shape)


In [0]:
print(df_restaurantes.dtypes)

print(df_menus.dtypes)

In [0]:
print(df_restaurantes.isnull().sum())

In [0]:
print(df_menus.isnull().sum())

## Parte 2 - Ingeniería de Características y Limpieza (Transform)
En df_restaurantes:
1. Mapeo de Precios: La columna rango_de_precios contiene símbolos ($, $$,
etc.). Utilice un diccionario y la función .map() o .replace() para transformarlos a
texto legible:
-$ -> "Económico"
-$$ -> "Moderadamente caro"
-$$$ -> "Caro"
-$$$$ -> "Muy caro"

In [0]:
df_restaurantes["price_range"] = df_restaurantes["price_range"].replace({
    "$": "Económico",
    "$$": "Moderadamente caro",
    "$$$": "Caro",
    "$$$$": "Muy caro"
})

In [0]:
df_restaurantes["price_range"].value_counts(dropna=False)

2. Desanidado de Ubicaciones (Split): 
La columna dirección_completa contiene la calle, ciudad, estado y código postal separados por comas. Utilice el método .str.split(',', expand=True) para crear tres nuevas columnas: calle, ciudad y estado_zip. Asegúrese de aplicar .str.strip() a la nueva columna ciudad para quitar espacios ocultos.


In [0]:
df_restaurantes[["calle", "ciudad", "estado_zip"]] = (
    df_restaurantes["full_address"]
    .str.split(",", expand=True, n=2)
)

In [0]:
df_restaurantes[
    ["full_address", "calle", "ciudad", "estado_zip"]
].head(10)

3. Filtro de Calidad: Elimine los restaurantes que no tengan puntaje (score nulo) o cuyo puntaje sea 0. Estandarice la columna category a minúsculas.


In [0]:
df_restaurantes = df_restaurantes[
    (df_restaurantes["score"].notna()) &
    (df_restaurantes["score"] != 0)
]

In [0]:
(df_restaurantes["score"] == 0).sum()

In [0]:
df_restaurantes["category"] = df_restaurantes["category"].str.lower()

In [0]:
df_restaurantes["category"].head()

En df_menus:
1. La columna price viene como texto (ej. "15,99 USD"). Cree una función lambda o use .str.replace() para eliminar el texto " USD", reemplazar la coma por un punto, y castear el resultado a formato decimal (float).

In [0]:
df_menus["price"] = (
    df_menus["price"]
    .str.replace(" USD", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

In [0]:
df_menus["price"].head(10)

2. Elimine los ítems del menú con precio igual a 0.0 o nulos

In [0]:
df_menus = df_menus[
    (df_menus["price"].notna()) &
    (df_menus["price"] != 0.0)
]

In [0]:
(df_menus["price"] == 0.0).sum()

### Optimización y Cruce (Control de Memoria RAM)
1. Para no colapsar la memoria del servidor, el CEO solo quiere analizar los restaurantes consolidados. Filtre df_restaurantes para quedarse únicamente con aquellos que tengan más de 100 calificaciones (ratings).

In [0]:
df_restaurantes = df_restaurantes[
    (df_restaurantes["ratings"].notna()) &
    (df_restaurantes["ratings"] > 100)
]

In [0]:
df_restaurantes.shape


2. Realice un INNER JOIN (pd.merge) entre su df_restaurantes filtrado y df_menus
utilizando el ID del restaurante. Guarde este cruce en un DataFrame llamado df_master.


In [0]:
df_master = pd.merge(
    df_restaurantes,
    df_menus,
    left_on="id",
    right_on="restaurant_id",
    how="inner"
)

In [0]:
df_master.head()

## Parte 3 - Carga a Base de Datos (Load)
1. Importe la librería sqlalchemy y genere un motor (create_engine) para una base de datos SQLite local llamada delivery_insights.db.


In [0]:
%pip install sqlalchemy

In [0]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///delivery_insights.db")

In [0]:
df_master.to_sql(
    name="master_food_data",
    con=engine,
    if_exists="replace",
    index=False
)

In [0]:
import pandas as pd

pd.read_sql(
    "SELECT * FROM master_food_data LIMIT 5",
    engine
)

## Parte 4 - Business Intelligence: Reporte al CEO (SQL + Visualización)


Conéctese a su base de datos SQLite mediante pd.read_sql(text(query), conn). Por cada pregunta, presente el Query ejecutado, el DataFrame resultante y un gráfico explicativo de alto nivel.
1. Penetración Geográfica: ¿Cuáles son las 5 ciudades (ciudad) con la mayor cantidad de restaurantes consolidados (los que quedaron en nuestra base)?

In [0]:
from sqlalchemy import text
import pandas as pd
import matplotlib.pyplot as plt

query = """
SELECT
    ciudad,
    COUNT(DISTINCT id) AS cantidad_restaurantes
FROM master_food_data
GROUP BY ciudad
ORDER BY cantidad_restaurantes DESC
LIMIT 5
"""

df_top_ciudades = pd.read_sql(
    text(query),
    engine
)

print(df_top_ciudades)

plt.figure(figsize=(8,5))

plt.barh(
    df_top_ciudades["ciudad"],
    df_top_ciudades["cantidad_restaurantes"]
)

plt.title("Top 5 ciudades con más restaurantes consolidados")
plt.xlabel("Cantidad de restaurantes")
plt.ylabel("Ciudad")

plt.gca().invert_yaxis()

plt.show()

2. Análisis de Oportunidad de Precio: El CEO cree que todos los restaurantes en el mercado son caros. Usando los datos, muestre la distribución de la columna rango_de_precios a nivel general (¿Cuántos hay en "Económico", "Caro", etc.?). Gráfico sugerido: Gráfico de torta (Pie chart) o Donut chart.

In [0]:
# Librerías
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text

# Query SQL
query = """
SELECT
    price_range AS rango_precio,
    COUNT(DISTINCT id) AS cantidad_restaurantes
FROM master_food_data
WHERE price_range IS NOT NULL
GROUP BY price_range
ORDER BY cantidad_restaurantes DESC
"""

# Ejecutar consulta
df_rango_precio = pd.read_sql(
    text(query),
    engine
)

# Mostrar DataFrame resultante
print(df_rango_precio)

# Crear Donut Chart
fig, ax = plt.subplots(figsize=(8, 8))

ax.pie(
    df_rango_precio["cantidad_restaurantes"],
    labels=df_rango_precio["rango_precio"],
    autopct="%1.1f%%",
    startangle=90
)

# Crear el agujero central
centro = plt.Circle((0, 0), 0.70, fc="white")
fig.gca().add_artist(centro)

# Título
plt.title("Distribución de Restaurantes por Rango de Precio")

plt.tight_layout()
plt.show()

3. Estrategia de Menú por Ciudad: Para las 3 ciudades más populares obtenidas en la pregunta 1, calcule el precio promedio de los platos en el menú. ¿En qué ciudad es más caro comer en promedio? Gráfico sugerido: Gráfico de barras comparativo.

In [0]:
# Librerías
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text

# Query SQL
query = """
SELECT
    ciudad,
    ROUND(AVG(price), 2) AS precio_promedio
FROM master_food_data
WHERE ciudad IN (
    SELECT ciudad
    FROM master_food_data
    GROUP BY ciudad
    ORDER BY COUNT(DISTINCT id) DESC
    LIMIT 3
)
GROUP BY ciudad
ORDER BY precio_promedio DESC
"""

# Ejecutar consulta
df_precio_ciudad = pd.read_sql(
    text(query),
    engine
)

# Mostrar DataFrame resultante
print(df_precio_ciudad)

# Gráfico de barras
plt.figure(figsize=(8, 5))

bars = plt.bar(
    df_precio_ciudad["ciudad"],
    df_precio_ciudad["precio_promedio"]
)

plt.title("Precio Promedio de los Platos en las 3 Ciudades Más Populares")
plt.xlabel("Ciudad")
plt.ylabel("Precio Promedio (USD)")

# Etiquetas sobre cada barra
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{bar.get_height():.2f}",
        ha="center",
        va="bottom"
    )

plt.show()

4. La Pregunta del Millón (Correlación Precio-Calidad): ¿Un restaurante más caro garantiza una mejor experiencia para el usuario? Escriba un query que calcule el puntaje (score) promedio agrupado por rango_de_precios. Utilice un gráfico de líneas o de barras para demostrar visualmente al CEO si la calidad sube a medida que aumenta la categoría de precio.

In [0]:
# Librerías
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text

# Query SQL
query = """
SELECT
    price_range AS rango_precio,
    ROUND(AVG(score), 2) AS score_promedio
FROM master_food_data
WHERE price_range IS NOT NULL
GROUP BY price_range
"""

# Ejecutar consulta
df_calidad_precio = pd.read_sql(
    text(query),
    engine
)

# Orden lógico de categorías
orden = [
    "Económico",
    "Moderadamente caro",
    "Caro",
    "Muy caro"
]

df_calidad_precio["rango_precio"] = pd.Categorical(
    df_calidad_precio["rango_precio"],
    categories=orden,
    ordered=True
)

df_calidad_precio = df_calidad_precio.sort_values("rango_precio")

# Mostrar DataFrame resultante
print(df_calidad_precio)

# Gráfico de líneas
plt.figure(figsize=(8,5))

plt.plot(
    df_calidad_precio["rango_precio"],
    df_calidad_precio["score_promedio"],
    marker="o",
    linewidth=2
)

plt.title("Relación entre Rango de Precio y Calidad Promedio")
plt.xlabel("Rango de Precio")
plt.ylabel("Score Promedio")

plt.grid(True, linestyle="--", alpha=0.5)

plt.show()